### Importing dependancies


In [ ]:
from typing import Tuple
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

### Defining feature groups

In [ ]:
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
BINARY_FEATURES = ["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling"]
NOMINAL_FEATURES = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod",
]
PASSTHROUGH_FEATURES = ["SeniorCitizen"]
DROP_COLUMNS = ["customerID"]
TARGET_COL = "Churn"

### Loading and splitting the data

In [ ]:
def load_and_split_data(
    csv_path: str = "telco.csv",
    test_size: float = 0.20,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Load raw dataset, enforce schema types, drop PII, and perform stratified split."""
    df = pd.read_csv(csv_path)

    # Drop non-predictive PII identifier (GDPR data minimization)
    df = df.drop(columns=DROP_COLUMNS)

    # Coerce whitespace strings in TotalCharges to NaN (imputed inside CV/train pipeline)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # Encode binary target
    y = df[TARGET_COL].map({"No": 0, "Yes": 1})
    X = df.drop(columns=[TARGET_COL])

    # Stratified 80/20 split BEFORE any fitting (Zero Data Leakage)
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )
    return X_train, X_test, y_train, y_test

### Adding preprocessing pipeline

In [ ]:
def build_preprocessor() -> ColumnTransformer:
    """Build deterministic ColumnTransformer matching the CM2604 data schema."""
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
            ("scaler", StandardScaler()),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("encoder", OrdinalEncoder(dtype=np.float64)),
        ]
    )

    nominal_transformer = Pipeline(
        steps=[
            (
                "onehot",
                OneHotEncoder(
                    drop="if_binary",
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, NUMERIC_FEATURES),
            ("bin", binary_transformer, BINARY_FEATURES),
            ("nom", nominal_transformer, NOMINAL_FEATURES),
            ("pass", "passthrough", PASSTHROUGH_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )
    return preprocessor

### Checking the pipeline

In [ ]:
def verify_pipeline(csv_path: str = "telco.csv") -> None:
    """Run deterministic assertions confirming shape parity and zero NaNs/leakage."""
    X_train, X_test, y_train, y_test = load_and_split_data(csv_path=csv_path)
    preprocessor = build_preprocessor()

    X_train_proc = preprocessor.fit_transform(X_train)
    X_test_proc = preprocessor.transform(X_test)

    assert X_train_proc.shape == (5634, 40), f"Unexpected train shape: {X_train_proc.shape}"
    assert X_test_proc.shape == (1409, 40), f"Unexpected test shape: {X_test_proc.shape}"
    assert np.isnan(X_train_proc).sum() == 0, "NaNs detected in processed training set"
    assert np.isnan(X_test_proc).sum() == 0, "NaNs detected in processed test set"
    assert np.isclose(y_train.mean(), y_test.mean(), atol=1e-4), "Stratification mismatch"

    print("Verification Passed:")
    print(f"  X_train_proc: {X_train_proc.shape} | Churn rate: {y_train.mean():.4f}")
    print(f"  X_test_proc:  {X_test_proc.shape} | Churn rate: {y_test.mean():.4f}")

In [ ]:
verify_pipeline()